# Beat SciNUP ensemble R@100 with fine-tuned BGE-base

**Target paper:** SciNUP (ECIR '26) — RRF ensemble row: R@100=0.4136, MAP=0.2163, MRR=0.6333, NDCG@10=0.4481  
**Our goal:** fine-tune `BAAI/bge-base-en-v1.5` on citation triplets, beat **R@100 > 0.4136**.

Before running:
1. Settings → Accelerator → **GPU T4 x2** (or any GPU), Internet → **ON**
2. Upload `hard_negative_triplets.csv` as a Kaggle Dataset (e.g. slug `scinup-triplets`) and attach it: Settings → Data → Add Input. Then set `TRIPLETS_KAGGLE_PATH` below.

In [ ]:
!pip install -q sentence-transformers rank-bm25
import torch
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
import os
os.makedirs('/kaggle/working/scinup', exist_ok=True)
!cd /kaggle/working/scinup && [ -f dataset.jsonl ] || (wget -q https://iai.group/downloads/SciNUP/dataset.tgz && tar xzf dataset.tgz && rm dataset.tgz sampled_users.tgz)
!cd /kaggle/working/scinup && [ -f qrels.trec ] || wget -q -O qrels.trec https://raw.githubusercontent.com/iai-group/SciNUP/main/data/retrieval_results/ground_truth_qrels.trec
!ls -la /kaggle/working/scinup/ && head -c 200 /kaggle/working/scinup/dataset.jsonl

In [ ]:
import glob
# Point this at your attached Kaggle dataset containing hard_negative_triplets.csv
TRIPLETS_KAGGLE_PATH = '/kaggle/input/scinup-triplets/hard_negative_triplets.csv'
cands = glob.glob('/kaggle/input/**/hard_negative_triplets.csv', recursive=True)
print('found:', cands)
TRIPLETS = cands[0] if cands else TRIPLETS_KAGGLE_PATH
assert os.path.exists(TRIPLETS), f'Missing triplets csv. Upload hard_negative_triplets.csv as a Kaggle Dataset and attach it. Looked at {TRIPLETS}'
print('using:', TRIPLETS)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

BASE = 'BAAI/bge-base-en-v1.5'
OUT = '/kaggle/working/bge-base-finetuned-citations'
EPOCHS, BS, LR, SEQ, LIMIT = 2, 16, 2e-5, 192, 20000

df = pd.read_csv(TRIPLETS)
if len(df) > LIMIT:
    df = df.sample(n=LIMIT, random_state=42).reset_index(drop=True)
print('triplets:', len(df))
examples = [InputExample(texts=[r.anchor_text, r.positive_text, r.negative_text]) for r in df.itertuples()]

model = SentenceTransformer(BASE)
model.max_seq_length = SEQ
loader = DataLoader(examples, batch_size=BS, shuffle=True)
loss = losses.MultipleNegativesRankingLoss(model=model, scale=20.0)
model.fit(train_objectives=[(loader, loss)], epochs=EPOCHS, warmup_steps=100,
          optimizer_params={'lr': LR}, show_progress_bar=True)
model.save(OUT)
print('saved:', OUT)

In [ ]:
import json
import numpy as np
from collections import defaultdict

def doc_text(c):
    return f"Title: {c['title'].strip()} Abstract: {c['abstract'].strip()}"

def load_records(path, limit=0):
    recs = [json.loads(l) for l in open(path, encoding='utf-8')]
    return recs[:limit] if limit else recs

def dense_run(records, model, top_k=100, bs=64):
    runs = {}
    for i, rec in enumerate(records):
        cands = rec['candidate_items']
        ids = [c['article_id'] for c in cands]
        V = model.encode([doc_text(c) for c in cands], batch_size=bs,
                         normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        q = model.encode([rec['nl_profile']], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)[0]
        runs[rec['author_id']] = list(zip(ids, map(float, V @ q)))
        if (i + 1) % 100 == 0:
            print(f'dense {i + 1}/{len(records)}', flush=True)
    return runs

def bm25_run(records):
    from rank_bm25 import BM25Okapi
    runs = {}
    for i, rec in enumerate(records):
        ids = [c['article_id'] for c in rec['candidate_items']]
        corpus = [doc_text(c).lower().split() for c in rec['candidate_items']]
        s = BM25Okapi(corpus).get_scores(rec['nl_profile'].lower().split())
        runs[rec['author_id']] = list(zip(ids, map(float, s)))
        if (i + 1) % 200 == 0:
            print(f'bm25 {i + 1}/{len(records)}', flush=True)
    return runs

def rrf(lists, k=60):
    out = {}
    for aid in lists[0]:
        sc = defaultdict(float)
        for runs in lists:
            for r, (doc, _) in enumerate(sorted(runs.get(aid, []), key=lambda x: x[1], reverse=True), 1):
                sc[doc] += 1.0 / (k + r)
        out[aid] = sorted(sc.items(), key=lambda x: x[1], reverse=True)
    return out

def read_trec(path, qrel):
    d = defaultdict(dict)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) < 4:
            continue
        d[p[0]][p[2]] = float(p[3]) if qrel else float(p[4])
    return d

def trec_metrics(run, qrels, k=100):
    R = M = RR = ND = n = 0.0
    cnt = 0
    for qid, gold in qrels.items():
        pos = {d for d, r in gold.items() if r > 0}
        if not pos:
            continue
        cnt += 1
        order = [d for d, _ in sorted(run.get(qid, {}).items(), key=lambda x: x[1], reverse=True)]
        hits = ap = first = 0
        for i, d in enumerate(order, 1):
            if d in pos:
                hits += 1
                if not first:
                    first = i
                ap += hits / i
        M += ap / len(pos)
        RR += 1.0 / first if first else 0.0
        R += sum(1 for d in order[:k] if d in pos) / len(pos)
        dcg = sum(1.0 / (2 ** r - 1) for r, d in enumerate(order[:10], 1) if d in pos)
        ideal = sum(1.0 / (2 ** r - 1) for r in range(1, min(10, len(pos)) + 1))
        ND += dcg / ideal if ideal else 0.0
    return {'n': cnt, f'R@{k}': R / cnt, 'MAP': M / cnt, 'MRR': RR / cnt, 'NDCG@10': ND / cnt}

def write_trec(path, runs, name, top_k=100):
    with open(path, 'w') as f:
        for aid, items in runs.items():
            seen, rank = set(), 1
            for doc, s in sorted(items, key=lambda x: x[1], reverse=True)[:top_k]:
                if doc in seen:
                    continue
                seen.add(doc)
                f.write(f'{aid} Q0 {doc} {rank} {s:.4f} {name}\n')
                rank += 1
print('helpers ok')

In [ ]:
records = load_records('/kaggle/working/scinup/dataset.jsonl')
print('authors:', len(records))
ft = SentenceTransformer(OUT)
ft.max_seq_length = 192
runs_ft = dense_run(records, ft, top_k=100, bs=64)
write_trec('/kaggle/working/bge_ft.trec', runs_ft, 'bge_ft')
print('wrote bge_ft.trec')

In [ ]:
runs_bm25 = bm25_run(records)
runs_rrf = rrf([runs_ft, runs_bm25])
write_trec('/kaggle/working/bge_rrf.trec', runs_rrf, 'bge_rrf')
qrels = read_trec('/kaggle/working/scinup/qrels.trec', True)
print(f"{'run':<10}{'R@100':>8}{'MAP':>8}{'MRR':>8}{'NDCG@10':>9}")
for name, r in [('bge_ft', runs_ft), ('bm25', runs_bm25), ('rrf', runs_rrf)]:
    m = trec_metrics({a: dict(v) for a, v in r.items()}, qrels)
    print(f"{name:<10}{m['R@100']:>8.4f}{m['MAP']:>8.4f}{m['MRR']:>8.4f}{m['NDCG@10']:>9.4f}  (n={m['n']})")
print('paper ensemble: R@100=0.4136 MAP=0.2163 MRR=0.6333 NDCG@10=0.4481')